In [1]:
import sys
import os
import mysql.connector
import pandas as pd
sys.path.append(os.path.abspath('..'))
from config import get_db_config
import warnings
warnings.filterwarnings('ignore')
import json
import pickle
import numpy as np
import datetime
from IPython.display import display, HTML

In [2]:
# Connect ke database config
config = get_db_config()
# Ambil host dari salah satu config (misal db_old)
print(f'Database config loaded: {config["db_old"]["host"]}')

# Connect ke DB Lama
db_old = mysql.connector.connect(**config['db_old'])
cursor_old = db_old.cursor(dictionary=True)
print(f'Connected to old database: {config["db_old"]["database"]}')

# Connect ke DB Baru
db_new = mysql.connector.connect(**config['db_new'])
cursor_new = db_new.cursor(dictionary=True)
print(f'Connected to new database: {config["db_new"]["database"]}')

db_future = mysql.connector.connect(**config['db_future'])
cursor_future = db_future.cursor(dictionary=True)
print(f'Connected to future database: {config["db_future"]["database"]}')


Database config loaded: localhost
Connected to old database: dataleap_v6_example
Connected to new database: 1
Connected to future database: 1


In [3]:
import pickle
import os

print("================================================================================")
print(" 🚀 SETUP INSERT HANDLER - FASE 1 (SISTEM MULTI-OWNER & AUTO-MAP) 🚀 ")
print("================================================================================")

# ================================================================================
# KONFIGURASI 1: LOAD MASING-MASING FILE PICKLE (TETAP TERPISAH)
# ================================================================================
data_cimut = {}
data_afrida = {}
data_hanif = {}

# 1. Load File cimut (Ganti nama file sesuai punyamu)
try:
    with open('fase_1_cimut.pkl', 'rb') as f:
        data_cimut = pickle.load(f)
    print("✓ Berhasil memuat data PKL milik cimut.")
except Exception as e:
    print(f"⚠️ Peringatan: Gagal memuat file pkl cimut: {e}")

# 2. Load File Afrida
try:
    with open('fase_1_afrida.pkl', 'rb') as f:
        data_afrida = pickle.load(f)
    print("✓ Berhasil memuat data PKL milik Afrida.")
except Exception as e:
    print(f"⚠️ Peringatan: Gagal memuat file pkl Afrida: {e}")

# 3. Load File Hanif
try:
    with open('fase_1_hanif.pkl', 'rb') as f:
        data_hanif = pickle.load(f)
    print("✓ Berhasil memuat data PKL milik Hanif.")
except Exception as e:
    print(f"⚠️ Peringatan: Gagal memuat file pkl Hanif: {e}")

print("\n================================================================================")
# ================================================================================
# KONFIGURASI 2: ISI DAFTAR TABEL MILIK MASING-MASING ORANG
# ================================================================================
list_table_cimut = [
    "users",
    "divisions",
    "shift_kerja",
    "admin_sarpras",
    "web_berita",
    "web_statistik",
]

list_table_afrida = [
    "kursus",
    "level",
    "sesi",
    "libur",
    "topik_diskusi",
    "kursus_level",
    "kursus_libur",
]

list_table_hanif = [
    "roles",
    "permissions",
    "role_has_permissions",
    "busdev_bidang",
    "syarat_resign",
    "ttd",
    "tag_siswa_keluar",
]

# ================================================================================
# KONFIGURASI 3: ATUR URUTAN MUTLAK PENYUNTIKAN KE DATABASE (MASTER ORDER)
# ================================================================================
# Masukkan nama tabel yang mau di-insert sesuai urutan FK (Foreign Key).
# Kamu bebas menyilangkan nama tabel di sini, sistem akan otomatis mencari pemiliknya.
master_urutan_insert = [
    "users",
    "divisions",
    "shift_kerja",
    "admin_sarpras",
    "web_berita",
    "web_statistik",
    "kursus",
    "level",
    "sesi",
    "libur",
    "topik_diskusi",
    "kursus_level",
    "kursus_libur",
    "roles",
    "permissions",
    "role_has_permissions",
    "busdev_bidang",
    "syarat_resign",
    "ttd",
    "tag_siswa_keluar",
]

# ================================================================================
# SISTEM DETEKTIF: MENCARI DAN MENGGABUNGKAN DATA BERDASARKAN PEMILIKNYA
# ================================================================================
print("🔍 Memulai proses pemetaan tabel ke pemilik masing-masing...\n")

data_siap_insert = {}

for table in master_urutan_insert:
    if table in list_table_cimut:
        data_siap_insert[table] = data_cimut.get(table)
        print(f"  📦 Tabel '{table}' otomatis dipetakan dari data cimut.")
        
    elif table in list_table_afrida:
        data_siap_insert[table] = data_afrida.get(table)
        print(f"  📦 Tabel '{table}' otomatis dipetakan dari data Afrida.")
        
    elif table in list_table_hanif:
        data_siap_insert[table] = data_hanif.get(table)
        print(f"  📦 Tabel '{table}' otomatis dipetakan dari data Hanif.")
        
    else:
        # Jika kamu memasukkan nama tabel di master_urutan tapi lupa memasukkannya di list pemilik
        data_siap_insert[table] = None
        print(f"  ❌ ERROR: Tabel '{table}' tidak ada di list cimut, Afrida, maupun Hanif!")

print("\n✅ Pemetaan selesai! Data siap disuntikkan ke database.")

 🚀 SETUP INSERT HANDLER - FASE 1 (SISTEM MULTI-OWNER & AUTO-MAP) 🚀 
✓ Berhasil memuat data PKL milik cimut.
✓ Berhasil memuat data PKL milik Afrida.
⚠️ Peringatan: Gagal memuat file pkl Hanif: (<StringDtype(storage='python', na_value=nan)>, array(['2026-06-21 19:26:28', '2026-06-21 19:26:28',
       '2026-06-21 19:26:28', '2026-06-21 19:26:28',
       '2026-06-21 19:26:28', '2026-06-21 19:26:28',
       '2026-06-21 19:26:28', '2026-06-21 19:26:28',
       '2026-06-21 19:26:28'], dtype=object))

🔍 Memulai proses pemetaan tabel ke pemilik masing-masing...

  📦 Tabel 'users' otomatis dipetakan dari data cimut.
  📦 Tabel 'divisions' otomatis dipetakan dari data cimut.
  📦 Tabel 'shift_kerja' otomatis dipetakan dari data cimut.
  📦 Tabel 'admin_sarpras' otomatis dipetakan dari data cimut.
  📦 Tabel 'web_berita' otomatis dipetakan dari data cimut.
  📦 Tabel 'web_statistik' otomatis dipetakan dari data cimut.
  📦 Tabel 'kursus' otomatis dipetakan dari data Afrida.
  📦 Tabel 'level' otomatis d

## Hide code

In [4]:
import pandas as pd
import datetime
import numpy as np
import mysql.connector

# ================================================================================
# TAHAP 3: FUNGSI UTAMA INSERT (ANTI SILENT-KILLER, AUTO-BATCHING & DIAGNOSTIC)
# ================================================================================
def insert_data_with_preview_and_skip_v2(db_connection, cursor, tables_data, ordered_list, batch_size=2000):
    results = {}
    
    print("="*80)
    print("🎬 MEMULAI EKSEKUSI PENYUNTIKAN DATA KE MYSQL BARU (SISTEM AUTO-BATCHING)")
    print("="*80)
    
    # ----------------------------------------------------------------------------
    # SUB-LANGKAH A: PROSES INSERT KE MYSQL DENGAN CHUNKING (LOOPING AMAN)
    # ----------------------------------------------------------------------------
    for table_name in ordered_list:
        if table_name not in tables_data:
            results[table_name] = {'status': 'not_found', 'msg': f'⚠️  {table_name}: Tidak ditemukan di file pkl', 'warnings': []}
            continue
            
        df_target = tables_data[table_name]
        
        if df_target is None or df_target.empty:
            results[table_name] = {'status': 'empty', 'msg': f'ℹ️  {table_name}: DataFrame kosong (0 baris)', 'warnings': []}
            continue
            
        try:
            # Bersihkan kolom kosong murni
            df_to_push = df_target.dropna(axis=1, how='all')
            columns_str = ', '.join([f'`{col}`' for col in df_to_push.columns])
            placeholders_str = ', '.join(['%s'] * len(df_to_push.columns))
            
            insert_query = f"INSERT IGNORE INTO `{table_name}` ({columns_str}) VALUES ({placeholders_str})"
            
            raw_numpy_list = df_to_push.to_numpy().tolist()
            
            # TUPLE GENERATOR (Mempertahankan "" untuk kolom Varchar NOT NULL)
            clean_data_tuples = [
                tuple(None if pd.isna(x) or str(x).strip() in ["NaT", "NaN"] else x for x in row) 
                for row in raw_numpy_list
            ]
            
            total_rows = len(clean_data_tuples)
            actual_inserted_total = 0
            db_warnings = []
            
            # 🔥 SISTEM AUTO-BATCHING (CHUNKING) 🔥
            # Loop memotong data menjadi bagian-bagian kecil agar MySQL tidak tersedak
            for i in range(0, total_rows, batch_size):
                chunk = clean_data_tuples[i : i + batch_size]
                cursor.executemany(insert_query, chunk)
                
                # Hitung data yang berhasil masuk pada batch ini
                chunk_inserted = max(0, cursor.rowcount)
                actual_inserted_total += chunk_inserted
                
                # Jika ada yang ter-skip di batch ini, tangkap errornya (maksimal simpan 3 per tabel)
                if chunk_inserted < len(chunk) and len(db_warnings) < 3:
                    cursor.execute("SHOW WARNINGS")
                    warnings_fetched = cursor.fetchall()
                    if warnings_fetched:
                        for w in warnings_fetched:
                            w_msg = f"MySQL Warning: {w['Message']}"
                            if w_msg not in db_warnings:
                                db_warnings.append(w_msg)
                            if len(db_warnings) >= 3:
                                break
                                
                # Commit per batch agar memori stabil
                db_connection.commit()
            
            # Evaluasi Status Akhir Tabel
            if actual_inserted_total == total_rows:
                status_flag = 'success'
                msg = f'✓ {table_name}: SEMPURNA! {total_rows}/{total_rows} baris sukses masuk database.'
            else:
                status_flag = 'partial_warning'
                msg = f'⚠️ {table_name}: TER-SKIP! Dikirim {total_rows} baris, tapi yang masuk DB HANYA {actual_inserted_total} baris.'

            results[table_name] = {
                'status': status_flag, 
                'msg': msg,
                'warnings': db_warnings
            }
            
        except Exception as e:
            db_connection.rollback()
            results[table_name] = {
                'status': 'failed', 
                'msg': f'✗ {table_name}: Gagal total saat eksekusi insert - Alasan: {e}',
                'warnings': []
            }

    # ----------------------------------------------------------------------------
    # 📊 CETAK PAPAN RINGKASAN DI PALING ATAS (SUMMARY BOARD)
    # ----------------------------------------------------------------------------
    print("\n================================================================================")
    print(" 📊 PAPAN RINGKASAN STATUS MIGRATION DATA (SUMMARY BOARD) 📊")
    print("================================================================================")
    
    print("🟢 TABEL YANG 100% SUKSES MASUK:")
    success_exist = False
    for table_name in ordered_list:
        res = results.get(table_name, {})
        if res.get('status') == 'success':
            print(f"  {res['msg']}")
            success_exist = True
    if not success_exist: print("  (Tidak ada tabel yang sukses sempurna)")

    print("\n🔴 TABEL YANG BERMASALAH / TER-SKIP (WAJIB DI CEK!):")
    failed_exist = False
    for table_name in ordered_list:
        res = results.get(table_name, {})
        if res.get('status') in ['failed', 'partial_warning', 'not_found', 'empty']:
            print(f"  {res['msg']}")
            
            # Cetak alasan dari MySQL (Dibatasi 3 agar tidak merusak tampilan Jupyter)
            if res.get('warnings'):
                for w_msg in res['warnings']:
                    print(f"      -> 🕵️ {w_msg}")
                    
            failed_exist = True
            
    if not failed_exist: print("  🎉 LUAR BIASA! Semua tabel bersih tidak ada data yang terbuang.")
            
    print("================================================================================\n")

    # ----------------------------------------------------------------------------
    # 📸 CETAK PREVIEW HISTORI & DIAGNOSTIK ERROR DI BAGIAN BAWAH
    # ----------------------------------------------------------------------------
    print("="*80)
    print("📸 MEMULAI LOG VISUALISASI PREVIEW & DIAGNOSTIK TABEL")
    print("="*80)
    
    for table_name in ordered_list:
        if table_name in results:
            res = results[table_name]
            
            if res['status'] == 'success':
                print(f"\n📂 [🟢 PREVIEW TABEL SUKSES: {table_name.upper()}]")
                print("-" * 50)
                display(tables_data[table_name].head(3))
                print("-" * 80)
                
            elif res['status'] in ['failed', 'partial_warning']:
                print(f"\n🚨 [🔴 DIAGNOSTIK TABEL ERROR: {table_name.upper()}] 🚨")
                print(f"Pesan Sistem: {res['msg']}")
                print("-" * 50)
                print("Berikut cuplikan data yang kemungkinan ditolak MySQL (Cek FK dan Tipe Data):")
                display(tables_data[table_name].head(5))
                print(f"\nTipe data Pandas untuk tabel '{table_name}':")
                print(tables_data[table_name].dtypes)
                print("-" * 80)
            
    print("\n" + "="*80)
    print("🏁 PROSES INSPEKSI SELESAI. SILAKAN CEK HASIL DIAGNOSTIK DI ATAS 🏁")
    print("="*80)
    return results

## Output

In [5]:
# ================================================================================
# TAHAP 4: MENJALANKAN EKSEKUSI DATA REAL
# ================================================================================
results_fase_1 = insert_data_with_preview_and_skip_v2(
    db_connection=db_future, 
    cursor=cursor_future, 
    tables_data=data_siap_insert,       # <--- Menggunakan data yang sudah di-mapping otomatis
    ordered_list=master_urutan_insert   # <--- Menggunakan urutan master buatanmu
)

🎬 MEMULAI EKSEKUSI PENYUNTIKAN DATA KE MYSQL BARU (SISTEM AUTO-BATCHING)

 📊 PAPAN RINGKASAN STATUS MIGRATION DATA (SUMMARY BOARD) 📊
🟢 TABEL YANG 100% SUKSES MASUK:
  ✓ users: SEMPURNA! 52/52 baris sukses masuk database.
  ✓ divisions: SEMPURNA! 7/7 baris sukses masuk database.
  ✓ shift_kerja: SEMPURNA! 3/3 baris sukses masuk database.
  ✓ admin_sarpras: SEMPURNA! 1/1 baris sukses masuk database.
  ✓ kursus: SEMPURNA! 20/20 baris sukses masuk database.
  ✓ level: SEMPURNA! 181/181 baris sukses masuk database.
  ✓ sesi: SEMPURNA! 43/43 baris sukses masuk database.
  ✓ libur: SEMPURNA! 78/78 baris sukses masuk database.
  ✓ topik_diskusi: SEMPURNA! 11/11 baris sukses masuk database.
  ✓ kursus_level: SEMPURNA! 177/177 baris sukses masuk database.
  ✓ kursus_libur: SEMPURNA! 1/1 baris sukses masuk database.

🔴 TABEL YANG BERMASALAH / TER-SKIP (WAJIB DI CEK!):
  ℹ️  web_berita: DataFrame kosong (0 baris)
  ℹ️  web_statistik: DataFrame kosong (0 baris)
  ℹ️  roles: DataFrame kosong (0 bari

,id_user,name,email,email_verified_at,password,remember_token,created_at,updated_at
0,U00001,ADMINISTRATOR,ditari@leapsurabaya.sch.id,2026-06-23 21:40:16.349224,aGtq,NaN,2026-06-23 21:40:16.349224,2026-06-23 21:40:16.349224
1,U00003,"Graciela Evanda Ronadi, S.Kom.",graciela@leapsurabaya.sch.id,2026-06-23 21:40:16.349224,qWmlbcVjYmo%3D,NaN,2026-06-23 21:40:16.349224,2026-06-23 21:40:16.349224
2,U00011,DANIAR AULIA RIZKI,daniar.rizki@leapsurabaya.sch.id,2026-06-23 21:40:16.349224,aGtq,NaN,2026-06-23 21:40:16.349224,2026-06-23 21:40:16.349224


--------------------------------------------------------------------------------

📂 [🟢 PREVIEW TABEL SUKSES: DIVISIONS]
--------------------------------------------------


,id_division,name_division,description,is_active
0,D00001,IT,None,1
1,D00002,Busdev,None,1
2,D00003,HR,None,1


--------------------------------------------------------------------------------

📂 [🟢 PREVIEW TABEL SUKSES: SHIFT_KERJA]
--------------------------------------------------


...

--------------------------------------------------------------------------------

📂 [🟢 PREVIEW TABEL SUKSES: KURSUS_LEVEL]
--------------------------------------------------


,id_kursus,id_level
0,K00001,L00001
1,K00001,L00002
2,K00001,L00003


--------------------------------------------------------------------------------

📂 [🟢 PREVIEW TABEL SUKSES: KURSUS_LIBUR]
--------------------------------------------------


,id_kursus,id_libur
1,K00001,L00068


--------------------------------------------------------------------------------

🏁 PROSES INSPEKSI SELESAI. SILAKAN CEK HASIL DIAGNOSTIK DI ATAS 🏁


In [6]:
# print("================================================================================")
# print(" 🧹 MEMULAI PROSES TRUNCATE DATA GLOBAL - FASE 1 (SISTEM RINGKASAN ATAS) 🧹 ")
# print("================================================================================")

# def truncate_tables_with_summary(db_connection, cursor, ordered_list):
#     truncate_results = {}
    
#     try:
#         # 🔥 SAKTI 1: Matikan benteng Foreign Key checks agar MySQL tidak memblokir penghapusan
#         cursor.execute("SET FOREIGN_KEY_CHECKS=0")
#         db_connection.commit()
#         print("🔓 Sensor Foreign Key Checks berhasil DIMATIKAN sementara.\n")
#     except Exception as e:
#         print(f"✗ Gagal mematikan Foreign Key Checks: {e}")
#         return
        
#     # ----------------------------------------------------------------------------
#     # SUB-LANGKAH A: PROSES EKSEKUSI TRUNCATE DI BELAKANG LAYAR
#     # ----------------------------------------------------------------------------
#     for table_name in ordered_list:
#         try:
#             truncate_query = f"TRUNCATE TABLE `{table_name}`"
#             cursor.execute(truncate_query)
#             db_connection.commit()
            
#             truncate_results[table_name] = {
#                 'status': 'success',
#                 'msg': f"✓ {table_name}: Sukses dibersihkan total! Seluruh baris data amblas."
#             }
#         except Exception as e:
#             db_connection.rollback()
#             truncate_results[table_name] = {
#                 'status': 'failed',
#                 'msg': f"✗ {table_name}: Gagal dikosongkan! Alasan: {e}"
#             }

#     try:
#         # 🔥 SAKTI 2: Wajib nyalakan kembali benteng Foreign Key checks setelah selesai
#         cursor.execute("SET FOREIGN_KEY_CHECKS=1")
#         db_connection.commit()
#         print("🔒 Sensor Foreign Key Checks berhasil DIHIDUPKAN kembali dengan aman.")
#     except Exception as e:
#         print(f"⚠️ Peringatan: Gagal menghidupkan kembali Foreign Key Checks: {e}")

#     # ----------------------------------------------------------------------------
#     # 🔥 CETAK PAPAN RINGKASAN TRUNCATE DI PALING ATAS (ANTI-SCROLL BOARD)
#     # ----------------------------------------------------------------------------
#     print("\n================================================================================")
#     print(" 📊 PAPAN RINGKASAN STATUS TRUNCATE DATABASE (CLEANUP SUMMARY BOARD) 📊")
#     print("================================================================================")
#     for table_name in ordered_list:
#         if table_name in truncate_results:
#             print(truncate_results[table_name]['msg'])
#         else:
#             print(f"⚠️  {table_name}: Lewat dari antrean pembersihan.")
#     print("================================================================================")
    
#     return truncate_results

# # === JALANKAN EKSEKUSI PEMBERSIHAN MENGGUNAKAN DAFTAR TABEL SALING SILANGMU ===
# results_truncate_fase_1 = truncate_tables_with_summary(
#     db_connection=db_new, 
#     cursor=cursor_new, 
#     ordered_list=tables_to_insert_ordered  # Otomatis memakai list urutan saling silang yang kita buat tadi
# )